In [1]:
from workflow_agent import LlmHandler

llm_handler = LlmHandler(
    llm_h_type="ollama",
    llm_h_params={
        "connection_string": "http://localhost:11434",
        "model_name": "gpt-oss:20b" #"llama3.1:latest" # "gemma3:27b"
    }
)

In [2]:
from workflow_agent import create_function_item

from typing import Type
from pydantic import BaseModel, Field

# --- Example usage ---

# Define mock functions and their associated Pydantic models:

# 1. get_weather function


class GetWeatherInput(BaseModel):
    city: str = Field(..., description="Name of the city for which weather to be extracted.")

class GetWeatherOutput(BaseModel):
    condition: str = Field(..., description="Weather condition in the requested city.")
    temperature: float = Field(..., description="Termperature in C in the requested city.")
    humidity: float = Field(None, description="Name of the city for which weather to be extracted.")

def get_weather(inputs: GetWeatherInput) -> GetWeatherOutput:
    """Get the current weather for a city from weather forcast api."""
    return GetWeatherOutput(
        condition = "Sunny",
        temperature = 20,
        humidity = 0.6
    )

# 2. send_report_email function

class EmailInformationPoint(BaseModel):
    title: str = Field(None, description="Few word description of the information.")
    content: str = Field(..., description="Content of the information.")

class SendReportEmailInput(BaseModel):
    city: str = Field(..., description="Name of the city where report will be send.")
    information: list[EmailInformationPoint]

class SendReportEmailOutput(BaseModel):
    email_sent: bool = Field(..., description="Conformation that email was send successfully.")
    message: str = Field(None, description="Optional comments from the process.")

def send_report_email(inputs: SendReportEmailInput) -> SendReportEmailOutput:
    """Sends a report email with given information points to a city."""
    return SendReportEmailOutput(
        email_sent = True,
        message = "Email sent to city of your choosing!"
    )

# 3. query_database function

class QueryDatabaseInput(BaseModel):
    topic: str = Field(..., description="Topic of a requested piece of information.")
    location: str = Field(None, description="Filter for location name.")
    uid: str = Field(None, description="Filter for unique indentifier of the database item.")

class QueryDatabaseOutput(BaseModel):
    info: str = Field(..., description="Content of the information.")
    uid: str = Field(None, description="Unique indentifier of the database item.")

def query_database(inputs : QueryDatabaseInput) -> QueryDatabaseOutput:
    """Get information from the database with provided filters."""
    return QueryDatabaseOutput(
        info = "Content extracted from the database for your query is ...",
        uid = "0000"
    )

# Create structured items for each function
available_functions = [
    create_function_item(get_weather, GetWeatherInput, GetWeatherOutput),
    create_function_item(send_report_email, SendReportEmailInput, SendReportEmailOutput),
    create_function_item(query_database, QueryDatabaseInput, QueryDatabaseOutput)
]

available_callables = {
    "get_weather" : get_weather,
    "send_report_email" : send_report_email,
    "query_database" : query_database
}

#### 0. Initialize Planner and Adaptor

In [3]:
from workflow_agent import WorkflowPlanner
import logging

wp = WorkflowPlanner(
    llm_h = llm_handler,
    available_functions=available_functions,
    loggerLvl = logging.DEBUG)

In [4]:
from workflow_agent import WorkflowAdaptor
from workflow_agent import InputCollector
import logging

wa = WorkflowAdaptor(
    llm_h = llm_handler, 
    input_collector_class = InputCollector,
    available_functions=available_functions,
    loggerLvl = logging.DEBUG)

In [5]:
from workflow_agent import WorkflowRunner

wr = WorkflowRunner(
    available_callables = available_callables, 
    available_functions = available_functions)

#### 1. Generating simple workflow using available functions (no input or output model)

In [6]:
task_description = """Query database to find information on birds and get latest weather for Berlin, then send an email there."""

planned_workflow_obj = await wp.generate_workflow(
    task_description=task_description)

planned_workflow_obj.workflow

[{'name': 'query_database', 'args': {'topic': 'birds'}},
 {'name': 'get_weather', 'args': {'city': 'Berlin'}},
 {'name': 'send_report_email',
  'args': {'city': 'Berlin',
   'information': [{'title': 'Bird Information',
     'content': 'source: query_database.output.info'},
    {'title': 'Weather Condition',
     'content': 'source: get_weather.output.condition'},
    {'title': 'Temperature',
     'content': 'source: get_weather.output.temperature'},
    {'title': 'Humidity', 'content': 'source: get_weather.output.humidity'}]}}]

In [7]:
adapted_workflow = await wa.adapt_workflow(
    workflow=planned_workflow_obj.workflow)

adapted_workflow

DEBUG:WorkflowAdaptor:step: get_weather adapted successfully on step 1! -------------------------------
DEBUG:WorkflowAdaptor:{"city":"Berlin","information":[{"title":"Bird Information","content":"1.output.info"},{"title":"Weather Condition","content":"2.output.condition"},{"title":"Temperature","content":"2.output.temperature"},{"title":"Humidity","content":"2.output.humidity"}]}
 Mapping for key 'content' is invalid.
 Mapping for array item at index 2 is invalid.
 Type mismatch in reference '2.output.humidity': expected target type 'string', got source type 'number'.
 Mapping for key 'content' is invalid.
 Mapping for array item at index 3 is invalid.
 Mapping for key 'information' is invalid.
DEBUG:WorkflowAdaptor:{"city":"Berlin","information":[{"title":"Bird Information","content":"1.output.info"},{"title":"Weather Condition","content":"2.output.condition"},{"title":"Temperature","content":"2.output.temperature"},{"title":"Humidity","content":"2.output.humidity"}]}
 Mapping for ke

[{'id': 1, 'name': 'query_database', 'args': {'topic': 'birds'}},
 {'id': 2, 'name': 'get_weather', 'args': {'city': 'Berlin'}},
 {'id': 3,
  'name': 'send_report_email',
  'args': {'city': 'Berlin',
   'information': [{'title': 'Bird Information', 'content': '1.output.info'},
    {'title': 'Weather Condition', 'content': '2.output.condition'},
    {'title': 'Temperature', 'content': 'Temperature unavailable'},
    {'title': 'Humidity', 'content': 'Humidity unavailable'}]}}]

#### 2. Generating simple workflow using available functions (no output model)

In [8]:
task_description_a = """Query database to find information on birds and get latest weather for the city, then send an email there."""

class WfInputs(BaseModel):
    city: str = Field(..., description="Name of the city for which weather to be extracted.")

planned_workflow_obj_a = await wp.generate_workflow(
    input_model = WfInputs,
    task_description=task_description_a)

planned_workflow_obj_a.workflow

[{'name': 'query_database', 'args': {'topic': 'birds'}},
 {'name': 'get_weather', 'args': {'city': 'source: WfInputs.output.city'}},
 {'name': 'send_report_email',
  'args': {'city': 'source: WfInputs.output.city',
   'information': [{'title': 'Bird Information',
     'content': 'source: query_database.output.info'},
    {'title': 'Weather Report',
     'content': 'source: get_weather.output.condition'}]}}]

In [9]:
adapted_workflow_a = await wa.adapt_workflow(
    workflow=planned_workflow_obj_a.workflow, 
    input_model = WfInputs)

adapted_workflow_a

DEBUG:WorkflowAdaptor:step: send_report_email adapted successfully on step 1! -------------------------------
DEBUG:WorkflowAdaptor:step: query_database adapted successfully on step 1! -------------------------------
DEBUG:WorkflowAdaptor:step: get_weather adapted successfully on step 1! -------------------------------
DEBUG:InputCollector:og_leaves : {'[0].args.topic': 'birds', '[1].args.city': 'source: WfInputs.output.city', '[2].args.city': 'source: WfInputs.output.city', '[2].args.information[0].title': 'Bird Information', '[2].args.information[0].content': 'source: query_database.output.info', '[2].args.information[1].title': 'Weather Report', '[2].args.information[1].content': 'source: get_weather.output.condition'}
DEBUG:InputCollector:mod_leaves : {'[0].id': '1', '[0].args.topic': 'birds', '[1].id': '2', '[1].args.city': '0.output.city', '[2].id': '3', '[2].args.city': '0.output.city', '[2].args.information[0].title': 'Bird Information', '[2].args.information[0].content': '1.ou

[{'id': 1, 'name': 'query_database', 'args': {'topic': 'birds'}},
 {'id': 2, 'name': 'get_weather', 'args': {'city': '0.output.city'}},
 {'id': 3,
  'name': 'send_report_email',
  'args': {'city': '0.output.city',
   'information': [{'title': 'Bird Information', 'content': '1.output.info'},
    {'title': 'Weather Report', 'content': '2.output.condition'}]}}]

#### 3. Generating simple workflow using available functions

In [6]:
task_description_b = """Query database to find information on birds and get latest weather for the city, then send an email there."""

class WfInputs(BaseModel):
    city: str = Field(..., description="Name of the city for which weather to be extracted.")

class WfOutputs(BaseModel):
    city: str = Field(..., description="Name of the city for which weather was extracted.")
    information: list[EmailInformationPoint] = Field(default=[], description="Information sent via email.")

planned_workflow_obj_b = await wp.generate_workflow(
    input_model = WfInputs,
    output_model = WfOutputs,
    task_description=task_description_b)

planned_workflow_obj_b.workflow

[{'name': 'query_database',
  'args': {'topic': 'birds', 'location': 'source: input_model.output.city'}},
 {'name': 'get_weather', 'args': {'city': 'source: input_model.output.city'}},
 {'name': 'send_report_email',
  'args': {'city': 'source: input_model.output.city',
   'information': [{'title': 'Birds Info',
     'content': 'source: query_database.output.info'},
    {'title': 'Weather Update',
     'content': 'source: get_weather.output.condition'}]}}]

In [7]:
adapted_workflow_b = await wa.adapt_workflow(
    workflow=planned_workflow_obj_b.workflow, 
    output_model = WfOutputs,
    input_model = WfInputs)

adapted_workflow_b

DEBUG:WorkflowAdaptor:{"topic":"birds","location":"source: 0.output.city"}
 Mapping for key 'location' is invalid.
DEBUG:WorkflowAdaptor:step: get_weather adapted successfully on step 1! -------------------------------
DEBUG:WorkflowAdaptor:step: output_model adapted successfully on step 1! -------------------------------
DEBUG:WorkflowAdaptor:step: send_report_email adapted successfully on step 1! -------------------------------
DEBUG:WorkflowAdaptor:step: query_database adapted successfully on step 2! -------------------------------
DEBUG:InputCollector:og_leaves : {'[0].args.topic': 'birds', '[0].args.location': 'source: input_model.output.city', '[1].args.city': 'source: input_model.output.city', '[2].args.city': 'source: input_model.output.city', '[2].args.information[0].title': 'Birds Info', '[2].args.information[0].content': 'source: query_database.output.info', '[2].args.information[1].title': 'Weather Update', '[2].args.information[1].content': 'source: get_weather.output.cond

[{'id': 1,
  'name': 'query_database',
  'args': {'topic': 'birds', 'location': '0.output.city'}},
 {'id': 2, 'name': 'get_weather', 'args': {'city': '0.output.city'}},
 {'id': 3,
  'name': 'send_report_email',
  'args': {'city': '0.output.city',
   'information': [{'title': 'Birds Info', 'content': '1.output.info'},
    {'title': 'Weather Update', 'content': '2.output.condition'}]}},
 {'id': '4',
  'name': 'output_model',
  'args': {'city': '0.output.city',
   'information': [{'title': 'Birds Info', 'content': '1.output.info'},
    {'title': 'Weather Update', 'content': '2.output.condition'}]}}]

#### 4. Testing generated workflow

In [8]:
test_outputs = wr.run_workflow(
    workflow = adapted_workflow_b, 
    inputs = WfInputs(city = "Berlin"),
    output_model = WfOutputs)

test_outputs.outputs

{'0': WfInputs(city='Berlin'),
 '1': QueryDatabaseOutput(info='Content extracted from the database for your query is ...', uid='0000'),
 '2': GetWeatherOutput(condition='Sunny', temperature=20.0, humidity=0.6),
 '3': SendReportEmailOutput(email_sent=True, message='Email sent to city of your choosing!'),
 '4': WfOutputs(city='Berlin', information=[EmailInformationPoint(title='Birds Info', content='Content extracted from the database for your query is ...'), EmailInformationPoint(title='Weather Update', content='Sunny')])}

##### Resetting runner

In [9]:
class GetWeatherInput(BaseModel):
    city: str = Field(..., description="Name of the city for which weather to be extracted.")

class GetWeatherOutput(BaseModel):
    condition: str = Field(..., description="Weather condition in the requested city.")
    temperature: float = Field(..., description="Termperature in C in the requested city.")
    humidity: float = Field(None, description="Name of the city for which weather to be extracted.")

def get_weather(inputs: GetWeatherInput) -> GetWeatherOutput:
    """Get the current weather for a city from weather forcast api."""
    error
    return GetWeatherOutput(
        condition = "Sunny",
        temperature = 20,
        humidity = 0.6
    )

def get_weather_report(inputs: GetWeatherInput) -> GetWeatherOutput:
    """Get the current weather for a city from weather forcast api (Improved)."""
    return GetWeatherOutput(
        condition = "Sunny",
        temperature = 20,
        humidity = 0.6
    )

In [10]:
# Create structured items for each function
available_functions2 = [
    create_function_item(get_weather, GetWeatherInput, GetWeatherOutput),
    create_function_item(get_weather_report, GetWeatherInput, GetWeatherOutput),
    create_function_item(send_report_email, SendReportEmailInput, SendReportEmailOutput),
    create_function_item(query_database, QueryDatabaseInput, QueryDatabaseOutput)
]

available_callables2 = {
    "get_weather" : get_weather,
    "get_weather_report" : get_weather_report,
    "send_report_email" : send_report_email,
    "query_database" : query_database
}

In [11]:
from workflow_agent import WorkflowPlanner

wr2 = WorkflowRunner(
    available_callables = available_functions2, 
    available_functions = available_callables2,
    loggerLvl = logging.DEBUG)

In [12]:
test_outputs2 = wr2.run_workflow(
    workflow = adapted_workflow_b, 
    available_functions = available_functions2,
    available_callables = available_callables2,
    inputs = WfInputs(city = "Berlin"),
    output_model = WfOutputs)

test_outputs2.outputs

{'0': WfInputs(city='Berlin'),
 '1': QueryDatabaseOutput(info='Content extracted from the database for your query is ...', uid='0000')}

In [16]:
planned_workflow_obj_b.model_dump()

{'retries': 0,
 'workflow': [{'name': 'query_database',
   'args': {'topic': 'birds', 'location': 'source: input_model.output.city'}},
  {'name': 'get_weather', 'args': {'city': 'source: input_model.output.city'}},
  {'name': 'send_report_email',
   'args': {'city': 'source: input_model.output.city',
    'information': [{'title': 'Birds information',
      'content': 'source: query_database.output.info'},
     {'title': 'Weather update',
      'content': 'source: get_weather.output.condition'}]}},
  {'id': 4, 'name': 'output_model'}],
 'init_messages': [{'role': 'system',
   'content': '## Role\nYou are a Workflow Agent tasked with creating a complete workflow for a given task.  Your workflow must be constructed solely from calls to the functions provided. Each workflow should be represented as a JSON list, where each element is an object representing a single function call. For any function input that is meant to be filled using the output of a previous step rather than provided direc

In [20]:
test_outputs2.error.model_dump()

{'error_message': 'Traceback (most recent call last):\n  File "/home/kyriosskia/miniforge3/envs/testenv/lib/python3.10/site-packages/workflow_agent/workflow_agent.py", line 567, in _run_func\n    output = func(inputs = inputs)\n  File "/tmp/ipykernel_18551/3203537686.py", line 11, in get_weather\n    error\nNameError: name \'error\' is not defined\n',
 'error_type': <WorkflowErrorType.RUNNER: 'runner'>,
 'additional_info': {'ffunction': 'get_weather'}}

In [17]:
print(test_outputs2.error.error_message)

Traceback (most recent call last):
  File "/home/kyriosskia/miniforge3/envs/testenv/lib/python3.10/site-packages/workflow_agent/workflow_agent.py", line 567, in _run_func
    output = func(inputs = inputs)
  File "/tmp/ipykernel_18551/3203537686.py", line 11, in get_weather
    error
NameError: name 'error' is not defined



In [18]:
test_outputs2.error.error_type

<WorkflowErrorType.RUNNER: 'runner'>

In [13]:
from workflow_agent import WorkflowPlanner
from workflow_agent import WorkflowAdaptor
from workflow_agent import InputCollector
import logging

wp2 = WorkflowPlanner(
    llm_h = llm_handler,
    available_functions=available_functions2,
    loggerLvl = logging.DEBUG)

wa2 = WorkflowAdaptor(
    llm_h = llm_handler, 
    input_collector_class = InputCollector,
    available_functions=available_functions2,
    loggerLvl = logging.DEBUG)


In [14]:
planned_workflow_obj_b.errors.append(test_outputs2.error)

planned_workflow_obj_tb = await wp2.generate_workflow(planned_workflow = planned_workflow_obj_b)

DEBUG:WorkflowPlanner:Attempt: 1


In [16]:
planned_workflow_obj_tb.workflow

[{'name': 'query_database',
  'args': {'topic': 'birds', 'location': 'source: input_model.output.city'}},
 {'name': 'get_weather_report',
  'args': {'city': 'source: input_model.output.city'}},
 {'name': 'send_report_email',
  'args': {'city': 'source: input_model.output.city',
   'information': [{'title': 'Birds Info',
     'content': 'source: query_database.output.info'},
    {'title': 'Weather Update',
     'content': 'source: get_weather_report.output.condition'}]}}]